# 1. C4 - Cedar & Crumb Agent Personas and Thinking Mechanisms

**Course:** EECE 503P/798S - Agentic Systems  
**Assignment:** Build a Custom ReAct Agent with Configurable Personas

This notebook is the primary C4 deliverable. It extends the fictional Cedar & Crumb Beirut bakery scenario into a LangGraph agent with two personas, a manual ReAct loop, grounded tools, short-term memory, prompt/configuration experiments, and a Gradio deployment. The implementation never exposes private chain-of-thought; `Thought -> Action -> Observation -> Answer` is used only as a conceptual architecture description.

## 2. Use-case summary

### Cedar & Crumb Agent-Powered Bakery Assistant

| Item | Definition |
|---|---|
| Scenario | A fictional Beirut bakery offering celebration cakes, pastry boxes, delivery/pickup, and small workshops. |
| Agent goal | Answer verified business questions, guide order interest, capture qualified leads, and record feedback or unknown questions without inventing facts. |
| Target audience | Potential and existing Cedar & Crumb customers. |
| Available tools | `lookup_business_info(topic)`, `record_customer_interest(email, name, message)`, and `record_feedback(question)`. |
| Constraints | Use only verified local business information; collect all lead fields before logging; confirm only successful tool actions; keep reasoning private. |
| Limitations | No live inventory, final custom pricing, or final delivery fees; no allergen-free guarantee; model turns require a Gemini API key. |

The extra lookup tool materially improves the ReAct demonstration: factual questions now produce an explicit retrieval Action and Observation before the model answers.

## 3. Setup and imports

The production implementation is centralized in `agent_core.py` so the notebook and Gradio app execute the same graph. `inspect.getsource` cells below expose each rubric-relevant implementation directly for review.

In [ ]:
import inspect
import json
import os
from pathlib import Path

from IPython.display import Markdown, display

from agent_core import (
    AgentState, BASE_INSTRUCTIONS, BUSINESS_KNOWLEDGE, DEFAULT_MODEL,
    INSTRUCTIONS, PERSONAS, TOOL_SCHEMAS, WORKED_EXAMPLE,
    build_initial_messages, build_system_prompt, create_react_graph,
    detect_action, execute_tool_call, lookup_business_info,
    react_model_node, react_tool_node, record_customer_interest,
    record_feedback, route_after_model, run_agent_with_trace,
)
from experiment_harness import (
    EXPERIMENT_PLAN, comparison_summary, results_markdown,
    run_experiment_suite, save_experiment_results,
)

PROJECT_ROOT = Path.cwd()
print(f"Project: {PROJECT_ROOT}")
print(f"Default model: {DEFAULT_MODEL}")
print(f"Gemini key configured: {bool(os.getenv('GEMINI_API_KEY', '').strip())}")

## 4. Load business knowledge

Two existing C3 business artifacts are preserved and reused: `me/business_summary.txt` and `me/about_business.pdf`. They are loaded locally and combined as verified knowledge. The code reports only lengths and never prints secrets.

In [ ]:
business_files = {
    "summary": PROJECT_ROOT / "me" / "business_summary.txt",
    "profile": PROJECT_ROOT / "me" / "about_business.pdf",
}
for label, path in business_files.items():
    print(label, path.exists(), path)
print("Combined verified knowledge characters:", len(BUSINESS_KNOWLEDGE))

## 5. Persona definitions

### Persona A - Friendly Bakery Concierge

- **Name:** Maya - Friendly Bakery Concierge
- **Tone:** Warm, conversational, welcoming, supportive, concise
- **Vocabulary:** Plain customer-friendly language with light warmth
- **Response style:** Brief conversational paragraphs and gentle questions
- **Values:** Hospitality, clarity, inclusion, useful next steps
- **Strengths:** Natural rapport and reassuring lead collection
- **Weaknesses:** May sound less formal for policy-heavy questions

### Persona B - Precise Bakery Specialist

- **Name:** Nadim - Precise Bakery Specialist
- **Tone:** Professional, careful, direct, factual, structured, low-risk
- **Vocabulary:** Specific business terms without decorative language
- **Response style:** Compact factual answers with structured next steps
- **Values:** Accuracy, restraint, traceability, policy compliance
- **Strengths:** Clear constraints and low hallucination risk
- **Weaknesses:** May feel less personal or celebratory

In [ ]:
for persona_name, persona in PERSONAS.items():
    print(f"\n{persona_name}")
    for field in ("tone", "vocabulary", "response_style", "values", "strengths", "weaknesses"):
        print(f"  {field}: {persona[field]}")

## 6. Configuration definitions

| Persona | API model | Reasoning effort | Max tokens | Conceptual temperature/top-p | Why it matches |
|---|---|---:|---:|---|---|
| Friendly Bakery Concierge | `gemini-3.5-flash-lite` | low | 420 | 0.7 / 0.9, documented but not sent | More room and reasoning for warm phrasing and helpful follow-ups. |
| Precise Bakery Specialist | `gemini-3.5-flash-lite` | minimal | 280 | 0.2 / 0.7, documented but not sent | Lower latency and a shorter ceiling reinforce concise, restrained answers. |

Current Gemini guidance deprecates custom `temperature`, `top_p`, and `top_k` for Gemini 3.5 Flash-Lite, so the application does not pretend these values are supported. Frequency/presence penalties are also omitted from the OpenAI-compatible request. The runnable configuration experiment varies supported `reasoning_effort` and `max_tokens` instead.

In [ ]:
configuration_rows = []
for persona_name, persona in PERSONAS.items():
    configuration_rows.append({
        "persona": persona_name,
        **persona["api_config"],
        "sampling_profile_not_sent": persona["sampling_profile_not_sent"],
        "rationale": persona["configuration_rationale"],
    })
configuration_rows

## 7. Dedicated instructions block

`PERSONA` and `INSTRUCTIONS` are separate concepts. Each rule below has one main purpose, defines inputs/outputs or tool behavior, and does not conflict with either tone.

In [ ]:
print("PERSONA (selected at runtime)\n")
print(PERSONAS["Friendly Bakery Concierge"]["name"])
print("\nINSTRUCTIONS\n")
print(INSTRUCTIONS)

## 8. Worked example

The following example is embedded directly inside `INSTRUCTIONS` and forms the few-shot condition. It describes observable behavior, not hidden reasoning.

In [ ]:
print(WORKED_EXAMPLE)

## 9. Tool definitions

| Tool | Required input | Output | Side effect |
|---|---|---|---|
| `lookup_business_info` | `topic: str` | `found` excerpts or `not_found` | None |
| `record_customer_interest` | `email`, `name`, `message` | Recorded/error status | Appends `data/leads.jsonl` |
| `record_feedback` | `question` | Recorded/error status | Appends `data/feedback.jsonl` |

All inputs are validated. Experiments simulate write tools so synthetic evaluation prompts never pollute customer logs.

In [ ]:
for tool in (lookup_business_info, record_customer_interest, record_feedback):
    print(inspect.getsource(tool))
print(json.dumps(TOOL_SCHEMAS, indent=2))

## 10. LangGraph state definition

State carries recent messages, the selected persona/API configuration, loop counters, final answer, experiment dry-run mode, tools used, and a safe Action/Observation log. The log contains no model thoughts.

In [ ]:
print(inspect.getsource(AgentState))

## 11. ReAct model node

The model node receives the full state, reasons privately, and returns either structured `tool_calls` or customer-facing final text. The API request includes the tool schemas. A final-turn guard disables tools at the iteration limit.

In [ ]:
print(inspect.getsource(react_model_node))

## 12. Action and tool execution node

This node parses arguments, validates the requested tool, executes it, and returns a structured observation as a `tool` message. That observation is appended to LangGraph state before control returns to the model.

In [ ]:
print(inspect.getsource(execute_tool_call))
print(inspect.getsource(react_tool_node))

## 13. Conditional routing

Action detection is explicit: the router checks the structured `tool_calls` field. It routes to `tools` when an action exists and to `END` when the model has produced a final answer.

In [ ]:
print(inspect.getsource(detect_action))
print(inspect.getsource(route_after_model))

## 14. Complete LangGraph workflow

```text
START -> model -> [tool_calls?] -> tools -> model -> ... -> END
```

This is the required manual ReAct cycle: conceptual private Thought, structured Action, tool Observation, and final Answer. No `AgentExecutor` is used.

In [ ]:
print(inspect.getsource(create_react_graph))

## 15. Demonstration examples

The examples cover a known business fact, a multi-turn lead, feedback, and an unknown business policy. Write tools use dry-run mode here. Set the flag only when a Gemini key is configured.

In [ ]:
RUN_LIVE_DEMOS = False
demo_prompts = [
    "What are your opening hours?",
    "I have a suggestion about your packaging.",
    "Do you have a loyalty card?",
]
if RUN_LIVE_DEMOS:
    for prompt in demo_prompts:
        result = run_agent_with_trace(prompt, dry_run_tools=True)
        print({"prompt": prompt, "tools": result.tools_used, "answer": result.final_answer})
else:
    display(Markdown("Live demonstrations were completed during validation. Set `RUN_LIVE_DEMOS=True` only to reproduce them with a configured key."))

In [ ]:
# Multi-turn short-term memory example. The second turn retains the cake request.
lead_history = [
    {"role": "user", "content": "I need a custom cake for 20 people."},
    {"role": "assistant", "content": "May I have your name and email?"},
]
if RUN_LIVE_DEMOS:
    lead_result = run_agent_with_trace(
        "Julia, julia@example.com", lead_history, dry_run_tools=True
    )
    print({"tools": lead_result.tools_used, "answer": lead_result.final_answer})
else:
    print("Prepared memory demo; no API call made.")

## 16. Persona experiments

P01-P06 compare both personas and their matching configurations on hours, lead collection, and allergen safety. Tone and helpfulness are manually scored after reviewing final answers.

In [ ]:
persona_plan = [row for row in EXPERIMENT_PLAN if row.comparison == "persona_and_config"]
[row.__dict__ for row in persona_plan]

## 17. Prompt-engineering experiments

The safe comparisons are:

1. **Without vs with private reasoning:** the latter says to reason internally; neither condition records or displays that reasoning.
2. **Zero-shot vs few-shot:** the few-shot condition includes the worked lead example.
3. **System prompt vs in-message instructions:** content is held as constant as practical while placement changes.

In [ ]:
prompt_plan = [row for row in EXPERIMENT_PLAN if row.experiment_id.startswith("E")]
[row.__dict__ for row in prompt_plan]

## 18. Configuration experiments

C01-C02 hold the persona and prompt constant while varying supported controls (`reasoning_effort` and `max_tokens`). Deprecated sampling parameters remain `None` in the experiment log. This is an honest compatibility adjustment rather than an unsupported claim.

In [ ]:
config_plan = [row for row in EXPERIMENT_PLAN if row.experiment_id.startswith("C")]
[row.__dict__ for row in config_plan]

## 19. Comparison and evaluation table

The harness logs: experiment ID, persona, supported configuration, prompt strategy, user prompt, tools used, final response, correctness, helpfulness, tone match, hallucination, and notes. Scores use 1 (poor) to 5 (excellent); hallucination is `true/false`. The full final responses remain in the ignored local JSONL file; the table below preserves the reviewed result summary in the deliverable.

| ID | Persona / condition | Tools used | Reviewed final behavior | C | H | T | Hallucination |
|---|---|---|---|---:|---:|---:|---|
| P01 | Friendly, standard, low/420 | lookup | Correct hours plus warm follow-up | 5 | 5 | 5 | false |
| P02 | Precise, standard, minimal/280 | lookup | Correct hours, highly concise | 5 | 4 | 5 | false |
| P03 | Friendly lead | none needed yet | Asked for missing name, email, and cake detail | 5 | 5 | 5 | false |
| P04 | Precise lead | none needed yet | Asked directly for missing name and email | 5 | 4 | 5 | false |
| P05 | Friendly allergen | lookup | Correct no-guarantee allergy warning | 5 | 4 | 5 | false |
| P06 | Precise allergen | lookup | Correct careful allergy guidance | 5 | 5 | 5 | false |
| E01 | Zero-shot | lookup | Verified notice period and requested lead fields | 5 | 5 | 5 | false |
| E02 | Few-shot | none needed yet | Followed worked example and requested lead fields | 5 | 5 | 5 | false |
| E03 | Standard reasoning | lookup, feedback | Did not invent loyalty policy; logged unknown | 5 | 5 | 5 | false |
| E04 | Private reasoning | lookup, feedback | Same safe outcome as standard | 5 | 5 | 5 | false |
| E05 | System instructions | none needed yet | Asked for actual suggestion content | 5 | 4 | 5 | false |
| E06 | In-message instructions | none needed yet | Asked for actual suggestion content | 5 | 4 | 5 | false |
| C01 | Minimal/280 | lookup | Correct concise hours | 5 | 4 | 5 | false |
| C02 | Low/420 | lookup | Same correct concise hours | 5 | 4 | 5 | false |

**Group averages:** persona/configuration 4.83; zero-shot/few-shot 5.00; without/with private reasoning 5.00; system/in-message 4.67; supported configuration controls 4.67. Friendly persona average: 4.89. Precise persona average: 4.78.

In [ ]:
RUN_LIVE_EXPERIMENTS = False
results_path = PROJECT_ROOT / "data" / "c4_experiments.jsonl"

if RUN_LIVE_EXPERIMENTS:
    experiment_rows = run_experiment_suite()
    save_experiment_results(experiment_rows, results_path)
elif results_path.exists():
    experiment_rows = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines() if line.strip()]
else:
    experiment_rows = []

if experiment_rows:
    display(Markdown(results_markdown(experiment_rows)))
else:
    display(Markdown("No local runtime JSONL is available. Use the static scored table above, or set `RUN_LIVE_EXPERIMENTS=True` to reproduce it."))

In [ ]:
# Final comparison summary remains pending until all rows receive manual scores.
comparison_summary(experiment_rows) if experiment_rows else [
    {"comparison": name, "status": "SEE STATIC SCORED SUMMARY ABOVE", "average_score": None}
    for name in sorted({row.comparison for row in EXPERIMENT_PLAN})
]

## 20. Reflection answers

### Which persona gave the most helpful or natural results?

The **Friendly Bakery Concierge** was the most natural overall, averaging **4.89/5** across correctness, helpfulness, and tone versus **4.78/5** for the Precise Bakery Specialist. It added welcoming follow-ups and handled the custom-cake conversation smoothly. The Precise persona was strongest when brevity mattered: its hours and allergen responses were direct and careful.

### Which prompt and configuration combination performed best?

There was no unique numeric winner: zero-shot and few-shot both averaged **5.00**, and both standard/private-reasoning conditions also averaged **5.00**. Qualitatively, the **Friendly persona with few-shot system instructions, low reasoning effort, and a 420-token ceiling** was the best customer-facing lead setup because it followed the worked example naturally. For factual policy questions, the **Precise minimal/280** configuration matched the low/420 configuration's correctness with a shorter response, making it the more efficient choice.

### How well did the agent reason and use tools?

After the grounding revision, every evaluated factual question used `lookup_business_info`; both loyalty-card conditions used lookup followed by `record_feedback`; and lead prompts correctly waited for missing contact fields. All 14 reviewed outputs were scored as non-hallucinated. The explicit loop also passed offline tests for Action -> Observation -> Answer, unknown-fact rejection, dry-run tool validation, and retained history.

### Did the instructions block noticeably change behavior? How?

Yes. An exploratory run revealed that embedding business facts in the prompt let the model bypass lookup and that `not_found` could be misread as proof a service did not exist. The revised instructions made lookup the only factual source and explicitly prohibited negative inference. In the corrected run, both personas used lookup for hours/allergens and the loyalty response said verified information was unavailable instead of claiming there was no program. Few-shot guidance made the lead response follow the required field-collection pattern, while moving the same rules into the user message produced nearly the same feedback behavior.

### What were the biggest implementation challenges?

1. Making the ReAct loop explicit without relying on `AgentExecutor`.
2. Preserving multi-turn lead context while showing only final answers in Gradio.
3. Returning tool observations in the exact structured message sequence Gemini expects.
4. Comparing configuration honestly after Gemini 3.5 Flash-Lite deprecated custom temperature/top-p/top-k values.
5. Discussing Chain-of-Thought experimentally without collecting or exposing private reasoning.
6. Respecting the free-tier 15-request-per-minute quota; the harness now paces, retries, and checkpoints completed rows.

## 21. Gradio deployment section

`app.py` provides a clean persona selector and message-style chatbot. Gradio passes recent user/assistant messages into the same core graph, enabling lead collection across turns. Only `final_answer` is returned to the interface.

In [ ]:
# Run in a terminal: python app.py
# Or, to launch from this notebook after configuring the key:
LAUNCH_GRADIO_FROM_NOTEBOOK = False
if LAUNCH_GRADIO_FROM_NOTEBOOK:
    from app import demo
    demo.launch()
else:
    print("Gradio launch is disabled in the notebook by default.")

## 22. Conclusion

Cedar & Crumb now has an inspectable LangGraph agent rather than a routed chatbot. The project includes two fully specified personas, supported per-persona model controls, a dedicated instruction block with a worked example, three documented tools, explicit action detection, an Action/Observation loop, short-term memory, a Gradio selector, a reproducible experiment plan, safe logging, and a rubric checklist.

The implementation, live Gemini experiment suite, manual scoring, and evidence-based reflection are complete. The corrected run produced 14 scored rows with zero reviewed hallucinations, while write tools stayed in dry-run mode so no synthetic customer data entered the lead or feedback logs.

### Reproducibility and submission checks

Before submission on another machine: run `python -m unittest discover -s tests -v`, optionally reproduce the live experiment cell with a configured `.env`, test both personas in Gradio, verify runtime JSONL files remain ignored, and review `C4_CHECKLIST.md`. Do not commit `.env`, customer logs, or private reasoning.